# MPLADS Risk Intelligence — Risk Scoring Pipeline
Runs on `MPLADS_final_selected.csv` (merged Recommended + Sanctioned + Completed, deduped on `WORK_RECOMMENDATION_DTL_ID`, coalesced to 22 clean columns).

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [2]:
df = pd.read_csv("MPLADS_final_selected.csv", low_memory=False)

In [3]:
df.shape

(103610, 22)

In [4]:
df.isnull().mean()*100

work_id                           0.000000
letter_no                         0.000000
mp_name                           0.000000
state_name                        0.000000
constituency                      0.000000
constituency_id                   0.000000
tenure                            0.000000
house_of_parliament               0.348422
ida_name                          0.000000
work_category                     0.000000
activity_name                     0.000000
work_description                  0.109063
work_stage                        0.481614
recommended_amount                0.348422
sanction_amount                   0.481614
actual_amount                    67.230962
recommendation_date               0.000000
sanction_date                    24.493775
actual_end_date                  67.230962
appears_in_recommended_report     0.000000
appears_in_sanctioned_report      0.000000
appears_in_completed_report       0.000000
dtype: float64

In [5]:
df.head()

,work_id,letter_no,mp_name,state_name,constituency,constituency_id,tenure,house_of_parliament,ida_name,work_category,...,work_stage,recommended_amount,sanction_amount,actual_amount,recommendation_date,sanction_date,actual_end_date,appears_in_recommended_report,appears_in_sanctioned_report,appears_in_completed_report
0,817,LN/MP431/2024-2025/149,Rao Inderjit Singh,Haryana,GURGAON,138.0,18th Lok Sabha,2.0,HISAR(DEPUTY COMMISSIONER HISSAR_IDA),Normal/Others,...,NaN,1100000.0,NaN,NaN,13-Aug-2024,NaN,NaN,True,False,False
1,822,LN/MP341/2024-2025/138,Shri LS Tejasvi Surya,Karnataka,BANGALORE SOUTH,182.0,18th Lok Sabha,2.0,CHIKKAMAGALURU(DEPUTY COMMISSIONER CHIKMAGALUR),Normal/Others,...,NaN,496984.0,NaN,NaN,22-Aug-2024,NaN,NaN,True,False,False
2,826,LN/MP661/2024-2025/11,Amol Ramsing Kolhe,Maharashtra,SHIRUR,254.0,18th Lok Sabha,2.0,MUMBAI(DISTRICT COLLECTOR MUMBAI CITY_IDA),Normal/Others,...,NaN,1500000.0,NaN,NaN,30-Aug-2024,NaN,NaN,True,False,False
3,830,LN/MP18007/2024-2025/2,Putta Mahesh Kumar,Andhra Pradesh,ELURU,15.0,18th Lok Sabha,2.0,Kakinada(DISTRICT COLLECTOR KAKINADA_IDA),Normal/Others,...,NaN,2500000.0,NaN,NaN,09-Sep-2024,NaN,NaN,True,False,False
4,834,LN/MP313/2024-2025/94,Shri Tapir Gao,Arunachal Pradesh,ARUNACHAL EAST,29.0,18th Lok Sabha,2.0,LOWER SIANG(Deputy Commisioner Lower Siang_IDA),Normal/Others,...,NaN,800000.0,NaN,NaN,15-Sep-2024,NaN,NaN,True,False,False


In [6]:
df["state_name"].value_counts()

state_name
Uttar Pradesh                                   18128
Gujarat                                          8240
Madhya Pradesh                                   7246
Bihar                                            5815
Odisha                                           5672
Tamil Nadu                                       5655
West Bengal                                      5633
Telangana                                        5256
Maharashtra                                      4692
Karnataka                                        4612
Jharkhand                                        4345
Kerala                                           4314
Punjab                                           4282
Andhra Pradesh                                   3525
Rajasthan                                        3440
Chhattisgarh                                     2283
Assam                                            2276
Haryana                                          1884
Himachal Pradesh 

## Funnel Status
Derived from the three `appears_in_*_report` boolean flags produced during the merge.

In [7]:
def get_funnel_status(row):
    if row["appears_in_completed_report"] == True:
        return "Sanctioned-Recommended-Completed"
    elif row["appears_in_sanctioned_report"] == True and row["appears_in_recommended_report"] == True:
        return "Sanctioned-Recommended-Not Completed"
    elif row["appears_in_sanctioned_report"] == True and row["appears_in_recommended_report"] == False:
        return "Sanctioned-Not Recommended-Not Completed"
    else:
        return "Not Sanctioned-Recommended"


In [8]:
df["funnel_status"] = df.apply(get_funnel_status, axis=1)

In [9]:
df["funnel_status"].value_counts()

funnel_status
Sanctioned-Recommended-Not Completed        44047
Sanctioned-Recommended-Completed            33952
Not Sanctioned-Recommended                  25378
Sanctioned-Not Recommended-Not Completed      233
Name: count, dtype: int64

In [10]:
df["recommendation_date"] = pd.to_datetime(df["recommendation_date"], errors="coerce")
df["sanction_date"] = pd.to_datetime(df["sanction_date"], errors="coerce")


In [11]:
today = pd.Timestamp.now()
df["days_since_recommendation"] = (today - df["recommendation_date"]).dt.days
df["days_since_sanction"] = (today - df["sanction_date"]).dt.days


## Risk Flags

In [12]:
##SANCTION_OVERDUE and COMPLETION_OVERDUE
df["sanction_overdue"] = (
    (df["funnel_status"] == "Not Sanctioned-Recommended") &
    (df["days_since_recommendation"] > 75)
)

df["completion_overdue"] = (
    (df["funnel_status"].isin([
        "Sanctioned-Recommended-Not Completed",
        "Sanctioned-Not Recommended-Not Completed"
    ])) &
    (df["days_since_sanction"] > 365)
)


In [13]:
print("Sanction overdue:", df["sanction_overdue"].sum())
print("Completion overdue:", df["completion_overdue"].sum())


Sanction overdue: 12253
Completion overdue: 10599


In [14]:
# Convert amount columns to numeric
df["actual_amount"] = pd.to_numeric(df["actual_amount"], errors="coerce")
df["sanction_amount"] = pd.to_numeric(df["sanction_amount"], errors="coerce")
df["recommended_amount"] = pd.to_numeric(df["recommended_amount"], errors="coerce")


In [15]:
##COST_OVERRUN
# Baselined against recommended_amount, NOT sanction_amount — actual_amount and
# sanction_amount are mutually exclusive by funnel stage (sanction_amount goes
# null once a work is Completed), so recommended_amount is the only field
# present at every stage.
df["cost_overrun"] = (
    df["actual_amount"].notna()
    & df["recommended_amount"].notna()
    & (df["actual_amount"] > df["recommended_amount"] * 1.10)
)


## Work Categorization (for outlier comparison groups)

In [16]:
def simple_category(text):
    text = str(text).lower()
    
    # Order matters — check more specific terms first
    categories = [
        (["road", "culvert", "bridge", "footpath", "bus-shed", "bus stop", "pedestrian"], "Roads & Transport"),
        (["school", "classroom", "library", "laboratory", "smart board", 
          "educational", "book", "van and bus", "furniture and fixture"], "Education"),
        (["hospital", "health", "ambulance", "prosthetic", "wheel chair", 
          "hearing aid", "medical"], "Health & Medical"),
        (["tube-well", "borewell", "hand pump", "drain", "gutter", 
          "toilet", "bathroom", "irrigation", "water"], "Water & Sanitation"),
        (["lighting", "street light", "electricity distribution", "electrification"], "Electricity & Lighting"),
        (["cctv", "security"], "Public Safety"),
        (["anganwadi", "crèche", "creche", "child"], "Childcare"),
        (["sports", "playfield", "playground", "gym", "park"], "Sports & Recreation"),
        (["crematorium", "burial", "cremation", "temple"], "Cultural & Religious"),
        (["hearse", "vehicle"], "Vehicles"),
        (["community", "hall", "sitting area", "staircase", "bench", "public"], "Community Infrastructure"),
    ]
    
    for keywords, category in categories:
        if any(kw in text for kw in keywords):
            return category
    
    return "Other"

df["simple_category"] = df["activity_name"].apply(simple_category)
print(df["simple_category"].value_counts())


simple_category
Roads & Transport           27059
Electricity & Lighting      25480
Community Infrastructure    20422
Water & Sanitation          13545
Education                    7119
Sports & Recreation          3225
Other                        2174
Cultural & Religious         1876
Health & Medical             1364
Public Safety                 737
Childcare                     342
Vehicles                      267
Name: count, dtype: int64


In [17]:
df["amount_percentile"] = df.groupby(["state_name","simple_category"])["recommended_amount"].rank(pct=True)

In [18]:
##HIGH_COST_OUTLIER
df["high_cost_outlier"] = df["amount_percentile"] > 0.95


In [19]:
##MISSING DESCRIPTION
is_null = df["work_description"].isna()
is_blank = df["work_description"].astype(str).str.strip() == ""
df["missing_description"] = is_null | is_blank


In [20]:
df["sanction_amount_missing"] = df["sanction_amount"].isna()

## Risk Score, Tier, Reason

In [21]:
## RISK SCORE
def risk_score(row):
    score = 0
    # 1. Completion overdue
    if row["completion_overdue"] == True:
        score += 30

    # 2. Cost overrun - severity based
    if row["cost_overrun"] == True and pd.notna(row["actual_amount"]) and pd.notna(row["recommended_amount"]) and row["recommended_amount"] > 0:
        overrun_percent = ((row["actual_amount"] - row["recommended_amount"]) / row["recommended_amount"]) * 100
        if overrun_percent <= 20:
            score += 10
        elif overrun_percent <= 50:
            score += 18
        else:
            score += 25

    # 3. High cost outlier
    if row["high_cost_outlier"] == True:
        score += 15
    # 4. Missing description
    if row["missing_description"] == True:
        score += 10
    # 5. Sanction overdue
    if row["sanction_overdue"] == True:
        score += 15
    # 6. Sanction amount missing
    if row["sanction_amount_missing"] == True:
        score += 5
    return score

df["risk_score"] = df.apply(risk_score, axis=1)


In [22]:
## RISK TIER (fixed bands against the 0–100 scale, NOT recalibrated per year)
def risk_tier(score):
    if score == 0: return "No Risk"
    elif score <= 15: return "Low"
    elif score <= 30: return "Medium"
    elif score <= 45: return "High"
    else:
        return "Critical"

df["risk_tier"] = df["risk_score"].apply(risk_tier)


In [23]:
df["risk_tier"].value_counts()

risk_tier
No Risk     76841
Low         15082
Medium      10974
High          712
Critical        1
Name: count, dtype: int64

In [24]:
##Risk Reason
def risk_reason(row):
    reason=[]
    if row["risk_score"]>0:
        if row["completion_overdue"] == True:
              reason.append("Completion delayed beyond one year of sanction.")
        if row["cost_overrun"] == True:
              reason.append("Actual expenditure exceeds recommended amount by more than 10%.")
        if row["high_cost_outlier"] == True:
              reason.append("Recommended amount is unusually high as compared to similar works.")
        if row["missing_description"] == True:
              reason.append("Work description is missing or not available.")
        if row["sanction_overdue"] == True:
              reason.append("recommended work not sanctioned within 75 days.")
        if row["sanction_amount_missing"] == True:
              reason.append("Sanction amount missing despite work being sanctioned.")
        return " ".join(reason)
    else:
        return "No major risks."

df["risk_reason"] = df.apply(risk_reason, axis=1)


In [25]:
df.sort_values("risk_score", ascending=False).head(10)[["completion_overdue","cost_overrun","high_cost_outlier","missing_description","sanction_overdue","sanction_amount_missing","risk_score","risk_reason"]]

,completion_overdue,cost_overrun,high_cost_outlier,missing_description,sanction_overdue,sanction_amount_missing,risk_score,risk_reason
12982,True,False,True,True,False,False,55,Completion delayed beyond one year of sanction...
47748,True,False,True,False,False,False,45,Completion delayed beyond one year of sanction...
16794,True,False,True,False,False,False,45,Completion delayed beyond one year of sanction...
23049,True,False,True,False,False,False,45,Completion delayed beyond one year of sanction...
716,True,False,True,False,False,False,45,Completion delayed beyond one year of sanction...
31438,True,False,True,False,False,False,45,Completion delayed beyond one year of sanction...
49885,True,False,True,False,False,False,45,Completion delayed beyond one year of sanction...
5608,True,False,True,False,False,False,45,Completion delayed beyond one year of sanction...
34073,True,False,True,False,False,False,45,Completion delayed beyond one year of sanction...
34517,True,False,True,False,False,False,45,Completion delayed beyond one year of sanction...


## Isolation Forest (ML anomaly layer)

In [26]:
from sklearn.ensemble import IsolationForest

features = ["recommended_amount", "sanction_amount", "actual_amount",
            "amount_percentile", "days_since_recommendation", "days_since_sanction"]

ml_df = df[features].fillna(-1)

iso = IsolationForest(n_estimators=200, contamination=0.05, random_state=42)
df["anomaly_flag"] = iso.fit_predict(ml_df)
df["anomaly_score"] = iso.decision_function(ml_df)
df["anomaly_flag"] = df["anomaly_flag"].map({1: False, -1: True})


## Duplicate Work Detection (TF-IDF, blocked by MP)

In [27]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

df["possible_duplicate"] = False

for mp, group in df.groupby("mp_name"):
    descs = group["work_description"].fillna("").astype(str)
    if len(descs) < 2:
        continue
    tfidf = TfidfVectorizer().fit_transform(descs)
    sim = cosine_similarity(tfidf)
    np.fill_diagonal(sim, 0)
    flagged_idx = group.index[(sim > 0.85).any(axis=1)]
    df.loc[flagged_idx, "possible_duplicate"] = True

print("Possible duplicates flagged:", df["possible_duplicate"].sum())


Possible duplicates flagged: 13975


In [28]:
df.to_csv("MPLAD_cleaned_v2.csv", index=False)

In [29]:
df["risk_score"].describe()

count    103610.000000
mean          5.617363
std          10.389083
min           0.000000
25%           0.000000
50%           0.000000
75%          15.000000
max          55.000000
Name: risk_score, dtype: float64

In [30]:
df["risk_tier"].value_counts()

risk_tier
No Risk     76841
Low         15082
Medium      10974
High          712
Critical        1
Name: count, dtype: int64

In [31]:
df["cost_overrun"].value_counts()


cost_overrun
False    103610
Name: count, dtype: int64